[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/electrokinetics/electrophoresis.ipynb)

# Capillary Electrophoresis

This example computes how a band of dye travels along a straight channel between charged walls, carried by the liquid when a pressure drop drives it and when an electric field does, and how dyes of opposite charge part under the field, the principle of capillary electrophoresis. A charged wall draws a thin cloud of ions of the opposite charge into the liquid beside it, the electric double layer. A pressure drop pushes all of the liquid alike while the walls hold it back, so the liquid runs fastest in the middle of the channel and stands still at the walls, and draws a band of dye out into a parabola. A field pushes the liquid only where it is charged, in the thin clouds along the walls, which drag the rest along, so in a channel many Debye lengths wide the liquid moves as one plug and carries the band along whole; this is why capillary electrophoresis drives its samples with a field rather than a pump. A dye that carries a charge also drifts through the liquid under the field, so a positive and a negative dye released together part into two plugs, one ahead of the liquid and one behind it.

The algebra is planar spacetime joined with one more direction, `v`, for the volume a flow carries. A parcel of the solution is modelled as a point whose weight is its volume, offset along time by the charge it holds, and a drive as a field of bivectors, the pressure's gradient wedged with volume and the electric potential's with time; a drive's commutator with a parcel is the force on it. A dye molecule is modelled as a parcel of charge without volume, `charge * time`, a point at infinity along time as an ion is, so the whole drive pushes it by `drive.commutator(charge * time)`: a neutral dye feels no push, and the pressure drop passes over every dye, since a dye has no volume. On the triangles of the channel, the dye a vertex's cell gains from a triangle is the triangle's dye moving at its drift, read by that corner's hat gradient, a plane paired with a direction; a triangle's hat gradients add up to nothing, so what one cell gains another loses, and every dye keeps its total. [The electrokinetic transport notebook](electrokinetics.ipynb) builds the double layer and the coupled flow and current end to end, through channels whose width and charge vary along them; here the flow and current are solved again in the straight channel, and only the double layer's settling is taken from it.

In the notation of continuum transport, the dye's concentration $c$ is carried by the liquid's velocity $\mathbf u$ and drifts under the field $\mathbf E$ at its mobility $\mu$ by its valence $z$, while it spreads at its diffusivity $D$: $\partial_t c + \nabla\cdot\big((\mathbf u + z\mu\mathbf E)\,c\big) = D\nabla^2 c$. Between plates $2h$ apart, a pressure drop gives Poiseuille's parabola, $u = \tfrac32 U\,(1 - y^2/h^2)$ for a mean speed $U$, and a field the Helmholtz–Smoluchowski plug, $u = -\epsilon\zeta E/\eta$ outside layers a few Debye lengths thick at the walls, with $\zeta$ the potential at the wall.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import tee

import numpy as np

from numga import stack
from numga.sparse import spdiag
from examples.util.animation import show_animation
from examples.fluids.electrokinetics import core, render, scenarios
from examples.fluids.electrokinetics.numerical import solve_coupled

# The algebra the channel's flow solve computes in: planar spacetime, and the volume a flow carries, a direction of its
# own, degenerate, since nothing measures a volume against a length or a time.
ga, mv = core.ga, core.mv
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# A drive: the electric potential's gradient wedged with time, and the pressure's, wedged with volume.
Drive = ga.gatype.from_blades("xt yt xv yv")                                  # Bivector+
axis = mv.x.dual()                                                            # [] Point-
across = mv.y.dual()                                                          # [] Point-
time = -mv.t.dual()                                                           # [] Point-
# A unit of volume, the weight of a point.
volume = mv.v.dual()                                                          # [] Point-
# A pump, a unit pressure drop along the channel, and a battery, a unit field along it.
pump, battery = mv.xv, mv.xt                                                  # [] Drive+, Drive+
experiments = stack((pump, battery))                                          # [experiments] Drive+

## The flows across a straight channel

The channel is straight, its walls uniformly charged and many Debye lengths apart, so that the double layers are thin beside it; its strip of triangles comes from the scenario, closed on itself along its length so that it has walls but no ends. On each triangle a corner's hat rises from nothing on the facing side to one at the corner, and its gradient is the plane through that side, along time and at unit volume, over twice the triangle's area, with no metric. The metric enters with the material: a potential rising by one at a corner drives the field down its hat's gradient, `pushes = -(gradients ^ mv.t)`, the vacuum answers each drive with its excitation, its push on a unit charge swept through time, the permittivity being one, and each drive reading every other's excitation, gathered over the triangles, is the field's outflow from each vertex's cell. From these, `core.double_layer` settles the double layer, the clouds of ions in their Boltzmann distribution, and the next cells solve the liquid's flow and the ions' current under each drive together. A drive pushes a parcel by its commutator with it, so each drive does work on a flow by reading what the flow carries, `drive & (flows & parcels)`: the pump, `mv.xv`, reads the volume and the battery, `mv.xt`, the charge, which lies only in the clouds along the walls. The liquid adds a potential of its own, whose drive reads the channel's current as nothing, so that no charge piles up anywhere, and the stream function and that potential are solved together. The flow is linear in its drive, so each drive is scaled to move the liquid at a unit mean speed, the volume it carries along, read by the pump, `pump & flux`, over the channel's width. The next two cells take about ten seconds, most of it settling the double layer and solving the flows.

In [ ]:
topology, places, shifts, lower, upper, charge, wall_charges = scenarios.straight()
# Each vertex a place of unit volume; each edge from its tail to its head, across the seam a period further along.
vertices = (volume + axis * places[..., 0] + across * places[..., 1]).field()   # Point-[vertices]
edges = topology.d[0] * vertices + (axis * shifts).field()                       # Point-[edges]
sides = topology.boundary[2, 1] * edges[..., None]                               # [corners] Point-[faces]
# Twice a triangle's area: two of its sides joined, read through time at unit volume.
double_areas = ((sides[..., 0] & sides[..., 1]) & time) & volume                # Scalar-[faces]
# Each corner's hat gradient: the plane through the facing side, along time and at unit volume, over twice the area.
gradients = (time & volume & sides) / double_areas[..., None]                   # [corners] Plane+[faces]
# The integrals weight by each area's size, an ordinary number.
areas = double_areas.norm() / 2                                                  # Scalar+[faces]


def gathered(local: Scalar):
    """A form on each triangle's corners `[..., corners, corners] Scalar+[faces]`, gathered onto the vertices it joins,
    `Scalar+[..., vertices <- vertices]`."""
    corners = topology.incidence[2, 0]                                           # [corners] Scalar+[faces <- vertices]
    return (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=-1).sum(axis=-1)


# Each hat's drive, the excitation the vacuum answers it with, and the field's outflow from each vertex's cell, each
# drive reading every other's excitation; and the cells' areas, a third of each triangle around.
pushes = -(gradients ^ mv.t)                                                     # [corners] Drive+[faces]
excitations = pushes.commutator(time).regressive(time)                           # [corners] Flux-[faces]
outflow = gathered((pushes[..., :, None] & excitations[..., None, :]) * areas[..., None, None])
cells = ~topology.incidence[2, 0].sum(axis=0) * (areas / 3)                       # Scalar+[vertices]

In [ ]:
# The ion clouds over the channel: the double layer's potential, which the companion notebook settles end to end.
potential = core.double_layer(outflow, cells, mv.scalar(charge[..., None]).field(), scenarios.valences,
                              scenarios.bulk, scenarios.n_newton_steps)                        # Scalar+[vertices]
corners = topology.incidence[2, 0]                                                             # [corners] Scalar+[faces <- vertices]
valences, bulk, mobilities = scenarios.valences, scenarios.bulk, scenarios.mobilities          # [species] each
# The ions in each triangle, the mean of its corners'; a parcel is its volume offset along time by their charge.
concentrations = corners.mean(axis=0) * (bulk * (-valences * potential[..., None]).exp())      # [species] Scalar+[faces]
parcels = volume + (time * valences * concentrations).sum(axis=-1)                            # Point-[faces]
conductivity = (mobilities * valences ** 2 * concentrations).sum(axis=-1)                     # Scalar+[faces]
# What a stream function rising by one at a corner carries over the triangle, and what the ions conduct under the
# drive of a potential rising by one there.
flows = sides / 2                                                                              # [corners] Point-[faces]
conducts = pushes.commutator(time).regressive(time)                                            # [corners] Flux-[faces]
# A velocity read as the plane that takes its circulation along a path: its run along and across, read back. Each
# corner's velocity read so by every other's is its circulation around the other's cell, and over the cell's area its
# vorticity, which viscosity resists.
circulating = (stack((mv.x, mv.y)) * (stack((mv.x, mv.y)) & Point)).sum(axis=0)              # [] Plane+ <- Point-
hat_velocities = flows / areas[..., None]                                                      # [corners] Point-[faces]
circulation = gathered((circulating(hat_velocities[..., :, None]) & hat_velocities[..., None, :]) * areas[..., None, None])
viscous = scenarios.viscosity * circulation * spdiag(1 / cells) * circulation                   # Scalar+[vertices <- vertices]
# A potential's drive reads what a flow carries, and what the ions conduct under another potential's drive.
carried = flows & parcels[..., None]                                                           # [corners] Flux-[faces]
coupling = gathered(carried[..., :, None] & pushes[..., None, :])
conduction = gathered((pushes[..., :, None] & conducts[..., None, :]) * (conductivity * areas)[..., None, None])

In [ ]:
# Each drive's work on each flow, and its ions' current read by each potential's drive; the stream function and the
# potential the liquid adds, solved together, nothing flowing through the walls.
flow_load = (~corners * (experiments[..., None] & carried)).sum(axis=-1)                      # [experiments] Scalar+[vertices]
applied_current = conductivity * experiments.commutator(time).regressive(time)                 # [experiments] Flux-[faces]
current_load = (~corners * ((pushes & applied_current[..., None]) * areas[..., None])).sum(axis=-1)
stream, added = solve_coupled(viscous, coupling, conduction, flow_load, current_load, topology.interior[0], upper,
                              int(lower[0]))                                                   # [experiments] Scalar+[vertices] each
# The liquid's velocity in each triangle, and the whole drive there, the applied one and the one the liquid adds.
velocities = ((corners * stream[..., None]) * sides).sum(axis=-1) / (2 * areas)                # [experiments] Point-[faces]
field = experiments + ((corners * added[..., None]) * pushes).sum(axis=-1)                     # [experiments] Drive+[faces]
# Each experiment's mean speed: the volume the parcels carry along per length, read by the pump, over the width.
flux = (velocities.regressive(parcels) * areas).sites.sum() / scenarios.straight_length        # [experiments] Flux-
mean_speeds = (pump & flux) / (2 * scenarios.straight_half_width)                              # [experiments] Scalar+
# The flow is linear in its drive: each scaled to move the liquid at a unit mean speed.
velocities, field = velocities / mean_speeds, field / mean_speeds          # [experiments] Point-[faces], [experiments] Drive+[faces]
render.draw_profiles(topology, vertices, scenarios.n_straight_across, velocities);

The liquid's velocity along the channel against the place across it, at a unit mean speed, under the pressure drop, orange, and under the field, blue. The pressure drop's flow is Poiseuille's parabola, half again as fast as the mean in the middle and still at the walls. The field's flow is flat across the channel, a plug a little faster than the mean, and falls to nothing only within a few Debye lengths of each wall, in the clouds of counter-ions that the field drags along.

## Bands of dye

A band of dye is released across the channel in four runs: a neutral dye in the pressure drop's flow and in the field's, and a positive and a negative dye together in the field's. Each dye drifts at its mobility under the whole drive's push on its charge, through the liquid that carries it, and spreads slowly as it goes, down its own gradient by Fick's law: a map from the dye's gradient to the flux it drives, `spreading`, read by every hat's gradient, `gradients & spreading(gradients)`. The dye feels the applied drive and the potential the liquid adds, but not the double layer's own field across the channel, which would draw a charged dye towards the walls or push it away: in a channel this wide the layers are thin beside it, and the band hardly reaches into them. The cell after the next takes about ten seconds, most of it stepping the dye and drawing it.

In [ ]:
# The band's middle along the channel and its width; the dye's mobility, the same for either charge, and its
# diffusivity, that of a large molecule; each run's drive, by its place among the experiments, and its dye's charge;
# the time step, the frames after the first and the steps between frames, and a frame's duration in milliseconds.
band_middle, band_width = 20.0, 4.0
dye_mobility, dye_diffusivity = 0.4, 0.05
run_drives, dye_charges = np.array([0, 1, 1, 1]), np.array([0, 0, 1, -1])
dye_dt, n_dye_frames, dye_stride = 0.2, 100, 3
frame_duration_ms = 40

In [ ]:
# A gradient read along and across as the flux it drives down itself at a unit diffusivity: Fick's law.
spreading = (stack((axis, across)) * (Plane & stack((axis, across)))).sum(axis=0)  # [] Point- <- Plane+


def dyeing(velocities: Point, field: Drive, charges: np.ndarray, dye: Scalar) -> Iterator[Scalar]:
    """The dye over the channel `[...] Scalar+[vertices]`, this dye and then a frame every `dye_stride` steps, in each
    run `[...]`: carried by the liquid at its velocity `[...] Point-[faces]` and drifting under the whole drive `[...]
    Drive+[faces]` by the charge it holds `[...]`, while it spreads."""
    # A dye molecule is a parcel of charge without volume: the drive pushes it by its charge alone, and it drifts at its
    # mobility through the liquid that carries it.
    drifts = velocities + dye_mobility * field.commutator(charges * time)             # [...] Point-[faces]
    # How much of a triangle's dye each corner's cell gains per unit time, the drift read by the corner's hat gradient,
    # from each corner's share of the triangle's dye.
    towards = areas[..., None] * (gradients & drifts[..., None])                      # [..., corners] Scalar+[faces]
    carrying = gathered(towards[..., :, None] * np.full(3, 1 / 3))                    # Scalar+[..., vertices <- vertices]
    # The overlaps of the corners' hats over each triangle, whose sum over a vertex's triangles is its cell.
    overlaps = gathered(areas[..., None, None] * (np.ones((3, 3)) + np.eye(3)) / 12)
    # How much each cell loses to its neighbours as the dye spreads: each hat's gradient reading the flux another's
    # drives down itself.
    diffusion = gathered(areas[..., None, None] * (gradients[..., :, None] & spreading(gradients[..., None, :])))
    changing = dye_diffusivity * diffusion - carrying                                 # Scalar+[..., vertices <- vertices]
    # Each step halfway implicit: half a step forward from the dye before it, and half a step back from the dye after it.
    forward = overlaps / dye_dt - changing / 2                                        # Scalar+[..., vertices <- vertices]
    backward = (overlaps / dye_dt + changing / 2) * Scalar                            # (Scalar+ <- Scalar+)[..., vertices <- vertices]
    yield dye
    for _ in range(n_dye_frames):
        for _ in range(dye_stride):
            dye = backward.solve(forward * dye)
        yield dye


# A band of dye across the channel, a bell over the distance along it as the plane x reads it, the same in every run.
distances = mv.x & vertices                                                           # Scalar+[vertices]
band = np.ones(len(dye_charges)) * (-((distances - band_middle) / band_width).squared() / 2).exp()   # [runs] Scalar+[vertices]
animated_dyes, dyes = tee(dyeing(velocities[run_drives], field[run_drives], dye_charges, band))
show_animation(render.animate_dyes(topology, vertices, scenarios.n_straight_across, scenarios.straight_length,
                                   wall_charges, animated_dyes), "electrophoresis", frame_duration_ms)

The dye shades each strip from white, where there is none, to its full colour at the strength the band starts with, between walls coloured by their negative charge, green. At the top a neutral dye, blue, rides the pressure drop's flow; in the middle the same dye rides the field's electro-osmotic flow; at the bottom a positive dye, magenta, and a negative one, green, ride that flow together, dark where they overlap. The pressure drop's flow draws the band out into a parabola, its middle running ahead and the dye beside the walls left behind; the electro-osmotic flow carries it on whole, held back only in the thin layers along the walls; and the charged dyes part, the positive one running ahead of the liquid along the field and the negative one falling behind.

In [ ]:
# checks
# The pressure drop's flow peaks at half again its mean speed, Poiseuille's parabola.
fastest = (mv.x & velocities[0]).to_array().max()
np.testing.assert_allclose(fastest, 1.5, rtol=1e-2)
# Every run's dye keeps its total over the cells: what one cell gains, another loses.
*_, final_dye = dyes
totals = (cells * band).sites.sum()                                                    # [runs] Scalar+
np.testing.assert_allclose((cells * final_dye).sites.sum().to_array(), totals.to_array(), rtol=1e-12)
# The positive and the negative dye, carried alike by the liquid, part at twice the drift the field alone gives them:
# their mobility times the battery's push on a unit of charge, scaled to a unit mean speed.
travelled = (cells * distances * (final_dye - band)).sites.sum() / totals              # [runs] Scalar+
drift = dye_mobility * (mv.x & battery.commutator(time)) / mean_speeds[1]               # [] Scalar+
np.testing.assert_allclose((travelled[2] - travelled[3]).to_array(),
                           (2 * n_dye_frames * dye_stride * dye_dt * drift).to_array(), rtol=1e-3)